In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

In [ ]:
# Task 2: Write your code here:

df_food.head()

In [ ]:
# Task 3: Write your code here:

df_food.info()

In [ ]:
# Task 4: Write your code here:

df_food.describe()

In [ ]:
# Task 5: Write your code here:

# Price distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time ')
plt.xlabel('Delivery')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_clean = df_food.copy()

df_clean = df_clean.drop(columns=['Order_ID'])

df_clean

In [ ]:
# Task 2: Write your code here:

for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')


df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mode()[0])
df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mean())

In [ ]:
# Task 3: Write your code here:

# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler, LabelEncoder


categorical_cols = ['Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Weather']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

In [ ]:
# Task 5: Write your code here:

feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day','Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs' ]

standard_scaler = StandardScaler()
df_scaled = data_standard_scaled = standard_scaler.fit_transform(df_clean[feature_cols])
df_scaled = pd.DataFrame(df_scaled)


In [ ]:
# Task 6: Write your code here:
import seaborn as sns


def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df_scaled
y = df_clean['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error


mae_scores = []
rmse_scores = []

# Split -
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")



#Evaluate
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE:  ${mae:,.2f}")




#Averaged score across all folds
y_fold_pred = model.predict(X_test)
mae_scores.append(mean_absolute_error(y_test, y_pred))
mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: